<a href="https://colab.research.google.com/github/Lobnaait/SEARCH_Lobna_Tsetline_CMRI/blob/main/features%20merged.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import argparse
import os
import sys

import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

# ----------------------------------------------------------------------------
# Options
# ----------------------------------------------------------------------------
# BMI and BSA-indexed volumes (volume / body surface area) are deterministic
# functions of height, weight and the volumes that are already kept. They are
# dropped by default; set to True if you want them as extra columns.
KEEP_DERIVED_INDICES = False

# Khened's wall-thickness features exist only for patients 1-100 (the other
# files cover 1-150). They are NOT redundant, so they are kept, with NaN for
# patients 101-150. Set to True to drop every column with missing values.
DROP_INCOMPLETE_COLUMNS = False

# Minimum Pearson r for two columns to be accepted as "the same quantity".
# (Different segmentations of the same heart never agree perfectly; RV values
# are the noisiest.)
MIN_R = 0.80

FILES = {  # prefix -> (file name, separator)
    "K": ("Khened_features.csv", ","),
    "W": ("Wolterink_features.txt", ","),
    "I": ("Isensee_feature.csv", ","),
    "C": ("Cetin_features.csv", ","),
}

# ----------------------------------------------------------------------------
# Redundancy registry
# Columns are referred to as <prefix>_<original name>, prefixes as in FILES.
# Kept reference = Wolterink (native ml, all 150 patients).
# Each entry: (dropped column, expression of the kept quantity it duplicates,
#              relation, explanation)
# The dropped column must be ~ the expression (same unit after the conversion).
# ----------------------------------------------------------------------------
BSA = "((W_height * W_weight / 3600) ** 0.5)"  # Mosteller (matches Isensee exactly)
R = []  # list of (dropped, kept_expr, relation, note)


def add(dropped, kept_expr, relation, note, derived=False):
    R.append((dropped, kept_expr, relation, note, derived))


# --- demographics / labels: identical in all files --------------------------
for p in "KIC":
    add(f"{p}_height", "W_height", "identical", "same patient height")
    add(f"{p}_weight", "W_weight", "identical", "same patient weight")
    add(f"{p}_group", "W_group", "identical", "same diagnosis label")
add("I_BMI", "W_weight / (W_height / 100) ** 2", "derived",
    "BMI is a function of height and weight", derived=True)

# --- chamber / myocardium volumes (ml) ---------------------------------------
for st_k, st_w in [("LV", "LV"), ("RV", "RV"), ("MYO", "MYO")]:
    for ph in ("ED", "ES"):
        # Khened: same quantity, different segmentation model
        add(f"K_{ph}_{st_k}", f"W_{st_w}_{ph}", "same quantity",
            f"{ph} volume of {st_k} (ml)")
        # Cetin: pyradiomics volume in mm3 of the SAME masks as Khened (exact match)
        for vol in ("VoxelVolume", "MeshVolume"):
            add(f"C_{st_w}_{ph}_shape_{vol}", f"W_{st_w}_{ph} * 1000",
                "same quantity (mm3 -> ml)", f"{ph} volume of {st_w}")

# --- Isensee: dynamic volume extrema = EDV / ESV / EF of ventricles ------------
for v, st in [("LVC", "LV"), ("RVC", "RV")]:
    add(f"I_dyn_{v}_vmax", f"W_{st}_ED", "same quantity",
        f"max volume over the cycle = end-diastolic volume of {st}")
    add(f"I_dyn_{v}_vmin", f"W_{st}_ES", "same quantity",
        f"min volume over the cycle = end-systolic volume of {st}")
    add(f"I_dyn_{v}_dyn_EF", f"W_EF_{st}", "same quantity",
        f"(vmax-vmin)/vmax = ejection fraction of {st}")
for st in ("LV", "RV"):
    add(f"K_EF_{st}", f"W_EF_{st}", "same quantity", f"ejection fraction {st}")

# --- LV mass: mass = 1.05 g/ml * myocardial volume -----------------------------
add("I_ED_LVM_mass", "W_MYO_ED * 1.05", "same quantity (x density)",
    "LV mass ED = myocardial volume x 1.05 g/ml")
add("I_ES_LVM_mass", "W_MYO_ES * 1.05", "same quantity (x density)",
    "LV mass ES = myocardial volume x 1.05 g/ml")

# --- volume / BSA ------------------------------------------------------------
for v, st in [("LVC", "LV"), ("RVC", "RV"), ("LVM", "MYO")]:
    for ph in ("ED", "ES"):
        add(f"I_{ph}_{v}_volume_per_bsa", f"W_{st}_{ph} / {BSA}", "derived",
            f"indexed volume = {st} {ph} volume / BSA", derived=True)

# --- ratios (LV/RV vs RV/LV are the same information, inverted) --------------
add("K_ED_LV_RV", "1 / W_RV_LV_ED", "reciprocal", "LV/RV volume ratio ED")
add("K_ES_LV_RV", "1 / W_RV_LV_ES", "reciprocal", "LV/RV volume ratio ES")
add("K_ED_MYO_LV", "W_MYO_LV_ED", "same quantity", "myocardium/LV volume ED")
add("K_ES_MYO_LV", "W_MYO_LV_ES", "same quantity", "myocardium/LV volume ES")
add("I_dyn_ratio_vmin_LVC_RVC", "1 / W_RV_LV_ES", "reciprocal",
    "LV ESV / RV ESV")

# Not dropped on purpose (related but NOT the same measurement):
#  - Khened *_MWT_* vs Isensee *_LVM_*thickness: different summary statistics
#    (per-slice mean/std aggregated differently), r = 0.5-0.9 -> kept both.
#  - Isensee dyn_LVM_*: extremes of myocardial volume over the cycle, not the
#    volume at ED/ES -> kept.
#  - Isensee ED_RVC_size_apical / *_ratio_apical: apical-slice measures -> kept.
#  - Isensee circularity / circumference, dyn_*_vol_median/std/skew/kurtosis,
#    dyn_dt_*, Cetin ED_ES_duration and all radiomics: unique.


# ----------------------------------------------------------------------------
def load(indir):
    frames = {}
    for p, (fname, sep) in FILES.items():
        path = os.path.join(indir, fname)
        df = pd.read_csv(path, sep=sep)
        if df["pid"].duplicated().any():
            sys.exit(f"{fname}: duplicated pid")
        frames[p] = df
        print(f"{fname:26s} {df.shape[0]:4d} patients  {df.shape[1]:4d} columns")
    return frames


def build_merged(frames):
    """Outer-merge everything on pid; columns prefixed by source."""
    pref = []
    for p, df in frames.items():
        d = df.set_index("pid").add_prefix(f"{p}_")
        pref.append(d)
    return pd.concat(pref, axis=1).reset_index()  # outer join on pid


def check_relations(M, keep_derived):
    rows = []
    for dropped, kept_expr, rel, note, derived in R:
        if dropped not in M.columns:
            continue
        x = M[dropped]

        if x.dtype == object or str(x.dtype).startswith(("str", "string")):
            # For object type columns, directly access the column that is being compared against.
            # This avoids numexpr's ValueError when it encounters object dtypes.
            # It's assumed here that for object 'dropped' columns, 'kept_expr' is a simple column name.
            y = M[kept_expr]
            same = (x == y).where(x.notna() & y.notna()).dropna()
            rows.append(dict(dropped=dropped, replaced_by=kept_expr, relation=rel,
                             n=len(same), pearson_r=np.nan, median_ratio=np.nan,
                             exact_match=bool(same.all()), note=note, derived=derived))
            continue

        # For numerical types, M.eval() is used to calculate 'y'
        y = M.eval(kept_expr)
        ok = x.notna() & y.notna()
        r = np.corrcoef(x[ok], y[ok])[0, 1]
        ratio = (x[ok] / y[ok].replace(0, np.nan)).median()
        rows.append(dict(dropped=dropped, replaced_by=kept_expr, relation=rel,
                         n=int(ok.sum()), pearson_r=round(r, 4),
                         median_ratio=round(ratio, 4), exact_match=np.nan,
                         note=note, derived=derived))
    rep = pd.DataFrame(rows)
    return rep


def main(indir=None, outdir=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--indir", default=".")
    ap.add_argument("--outdir", default=".")
    # parse_known_args: ignores the extra '-f kernel.json' that Jupyter/Colab injects
    a, _ = ap.parse_known_args()
    if indir is not None:
        a.indir = indir
    if outdir is not None:
        a.outdir = outdir

    frames = load(a.indir)
    M = build_merged(frames)

    # sanity: labels must agree across files
    lab = M[[f"{p}_group" for p in FILES]]
    bad = lab.nunique(axis=1, dropna=True) > 1
    if bad.any():
        sys.exit(f"Group labels disagree for pids: {M.loc[bad, 'pid'].tolist()}")

    rep = check_relations(M, KEEP_DERIVED_INDICES)
    weak = rep[(rep.pearson_r < MIN_R)]
    for _, w in weak.iterrows():
        print(f"WARNING: {w.dropped} vs {w.replaced_by}: r={w.pearson_r} < {MIN_R}; "
              f"check before treating as redundant")

    # columns to drop
    to_drop = []
    for dropped, _, _, _, derived in R:
        if dropped not in M.columns:
            continue
        if derived and KEEP_DERIVED_INDICES:
            continue
        to_drop.append(dropped)

    out = M.drop(columns=to_drop)

    # Clean names: drop the source prefix (no collisions are left, checked below)
    new_names = {}
    for c in out.columns:
        if c == "pid":
            new_names[c] = c
        else:
            new_names[c] = c.split("_", 1)[1]
    if len(set(new_names.values())) != len(new_names):
        dup = pd.Series(list(new_names.values()))
        print("Name collision after removing prefixes, keeping prefixes for:",
              dup[dup.duplicated(keep=False)].unique().tolist())
        seen = pd.Series(list(new_names.values())).value_counts()
        new_names = {c: (n if seen[n] == 1 else c) for c, n in new_names.items()}
    out = out.rename(columns=new_names)

    # put identifiers first
    first = ["pid", "group", "height", "weight"]
    out = out[[c for c in first if c in out.columns] +
              [c for c in out.columns if c not in first]]

    if DROP_INCOMPLETE_COLUMNS:
        out = out.dropna(axis=1)

    out = out.sort_values("pid")
    os.makedirs(a.outdir, exist_ok=True)
    out_path = os.path.join(a.outdir, "all_features_merged.csv")
    rep_path = os.path.join(a.outdir, "redundancy_report.csv")
    out.to_csv(out_path, index=False)
    rep.to_csv(rep_path, index=False)

    n_in = sum(df.shape[1] for df in frames.values())
    print(f"\ncolumns in the 4 files (with repeats): {n_in}")
    print(f"redundant columns removed            : {len(to_drop)}")
    print(f"columns in merged file               : {out.shape[1]}  "
          f"({out.shape[0]} patients)")
    nan_cols = out.columns[out.isna().any()].tolist()
    if nan_cols:
        print(f"columns with missing values ({len(nan_cols)}), e.g. pid > 100: {nan_cols}")
    print(f"written: {out_path}\n         {rep_path}")


if __name__ == "__main__":
    main()

Khened_features.csv         100 patients    24 columns
Wolterink_features.txt      150 patients    16 columns
Isensee_feature.csv         150 patients    67 columns
Cetin_features.csv          150 patients   557 columns

columns in the 4 files (with repeats): 664
redundant columns removed            : 49
columns in merged file               : 612  (150 patients)
columns with missing values (8), e.g. pid > 100: ['ES_MWT_max_mean', 'ES_MWT_std_mean', 'ES_MWT_mean_std', 'ES_MWT_std_std', 'ED_MWT_max_mean', 'ED_MWT_std_mean', 'ED_MWT_mean_std', 'ED_MWT_std_std']
written: ./all_features_merged.csv
         ./redundancy_report.csv
